# Day 96: Dynamic Treatments: The g-Formula and Marginal Structural Models

*Beginner to Ninja: Math for Causal Inference*

When treatment is decided repeatedly over time, a covariate measured between decisions can be both a confounder of later treatment and a result of earlier treatment. Adjusting for it in a regression blocks part of the effect and can open collider paths; ignoring it leaves confounding. Robins's g-formula and inverse-probability-weighted marginal structural models handle this time-varying confounding correctly.

**Learning outcomes**

- State sequential exchangeability, positivity and consistency for treatment sequences.
- Explain why standard regression fails under treatment–confounder feedback.
- Derive and compute the g-formula for static and dynamic regimes.
- Fit a marginal structural model with (stabilised) inverse-probability-of-treatment weights.

## Theory

### Setting

Two decision points: treatment $A_0$, then a covariate $L_1$ (e.g. blood pressure at the next visit), then treatment $A_1$, then outcome $Y$. A **regime** fixes $(a_0, a_1)$ (static) or makes $a_1$ depend on $L_1$ (dynamic, e.g. "intensify treatment only if blood pressure is still high"). The target is $E[Y(\bar a)]$ for a regime $\bar a$.

### Treatment–confounder feedback

$L_1$ is affected by $A_0$ and affects $A_1$; it also shares an unmeasured cause $U$ (underlying frailty) with $Y$:

- Not adjusting for $L_1$: $A_1$ is confounded through $L_1 \leftarrow U \to Y$.
- Adjusting for $L_1$: blocks the part of $A_0$'s effect that runs through $L_1$, and opens the collider path $A_0 \to L_1 \leftarrow U \to Y$.

No single regression gets both treatment effects right.

### Identification

Assume **sequential exchangeability** $Y(\bar a) \perp A_0$ and $Y(\bar a) \perp A_1 \mid A_0, L_1$, **positivity** and **consistency**. Then the **g-formula** holds:

$$
E[Y(a_0, a_1)] = \sum_{l}E[Y \mid A_0 = a_0, L_1 = l, A_1 = a_1]\;P(L_1 = l \mid A_0 = a_0).
$$

For a dynamic regime $a_1 = d(l)$, replace $a_1$ by $d(l)$ inside the sum.

### Marginal structural models

Model the counterfactual means directly, e.g. $E[Y(a_0, a_1)] = \beta_0 + \beta_1a_0 + \beta_2a_1$, and fit by weighted least squares with

$$
W = \frac{1}{P(A_0 \mid \cdot)\,P(A_1 \mid A_0, L_1)}, \qquad SW = \frac{P(A_0)\,P(A_1 \mid A_0)}{P(A_0)\,P(A_1 \mid A_0, L_1)} \text{ (stabilised)}.
$$

Weighting creates a pseudo-population in which $A_1$ no longer depends on $L_1$, while $L_1$ remains free to carry the effect of $A_0$.

### Beyond two periods

With $T$ periods the g-formula integrates over the whole covariate history (usually by Monte Carlo simulation); MSMs then impose structure such as $E[Y(\bar a)] = \beta_0 + \beta_1\sum_t a_t$ ("cumulative treatment").

## Part 1 · Warm-up

### Exercise 1 · Is it a confounder or a mediator?

In the blood-pressure example, is $L_1$ a confounder, a mediator, or both?

<details><summary>Show solution</summary>

**Answer:** both: it mediates part of the effect of $A_0$ (treatment lowers blood pressure) and confounds $A_1$ (doctors intensify treatment when blood pressure is high).

</details>

*Your work:*

### Exercise 2 · g-formula by hand

$P(L_1 = 1 \mid A_0 = 1) = 0.25$; $E[Y \mid 1, L_1 = 1, 1] = 9.5$, $E[Y \mid 1, L_1 = 0, 1] = 7.5$. Compute $E[Y(1, 1)]$.

<details><summary>Show solution</summary>

**Step 1.** $0.25\times9.5 + 0.75\times7.5 = 2.375 + 5.625 = 8.0$.

</details>

*Your work:*

### Exercise 3 · A weight

A patient had $A_0 = 1$ (probability 0.5, randomised) and $A_1 = 1$ with $P(A_1 = 1 \mid A_0 = 1, L_1 = 0) = 0.28$. Compute the IPT weight.

<details><summary>Show solution</summary>

**Answer:** $W = 1/(0.5\times0.28) = 7.14$: patients treated against the typical pattern count for more.

</details>

*Your work:*

### Exercise 4 · Positivity

If doctors always intensify treatment when $L_1 = 1$, which counterfactual mean cannot be identified?

<details><summary>Show solution</summary>

**Answer:** any regime with $a_1 = 0$ requires $E[Y \mid a_0, L_1 = 1, A_1 = 0]$, which is never observed; $E[Y(a_0, 0)]$ is not identified.

</details>

*Your work:*

### Exercise 5 · Dynamic regime

Write the g-formula for "treat at baseline; intensify only if $L_1 = 1$".

<details><summary>Show solution</summary>

**Answer:** $P(L_1 = 1 \mid A_0 = 1)E[Y \mid 1, 1, 1] + P(L_1 = 0 \mid A_0 = 1)E[Y \mid 1, 0, 0]$.

</details>

## Part 2 · Proofs

*Your work:*

### Exercise 6 · Deriving the g-formula

Use sequential exchangeability and consistency to derive the g-formula.

<details><summary>Show solution</summary>

**Step 1.** $E[Y(a_0, a_1)] = E[Y(a_0, a_1) \mid A_0 = a_0]$ (exchangeability at time 0).

**Step 2.** $= \sum_lE[Y(a_0, a_1) \mid A_0 = a_0, L_1 = l]P(L_1 = l \mid A_0 = a_0)$, using consistency for $L_1 = L_1(a_0)$ when $A_0 = a_0$.

**Step 3.** Exchangeability at time 1 adds $A_1 = a_1$ to the conditioning; consistency replaces $Y(a_0, a_1)$ by $Y$. $\blacksquare$

</details>

*Your work:*

### Exercise 7 · IPTW identifies the counterfactual mean

Show $E\Big[\frac{\mathbf 1\{A_0 = a_0, A_1 = a_1\}Y}{P(a_0)P(a_1 \mid a_0, L_1)}\Big] = E[Y(a_0, a_1)]$ when $A_0$ is randomised.

<details><summary>Show solution</summary>

**Step 1.** Condition on $(A_0, L_1)$: $E[\mathbf 1\{A_1 = a_1\}Y \mid a_0, L_1] = P(a_1 \mid a_0, L_1)E[Y \mid a_0, L_1, a_1]$.

**Step 2.** Dividing by $P(a_1 \mid a_0, L_1)$ leaves $E[Y \mid a_0, L_1, a_1]$; then $E[\mathbf 1\{A_0 = a_0\}(\cdot)/P(a_0)]$ averages it over $P(L_1 \mid a_0)$.

**Step 3.** The result is the g-formula. $\blacksquare$

</details>

*Your work:*

### Exercise 8 · Why adjusting for L₁ biases A₀'s effect

In the DAG $A_0 \to L_1 \leftarrow U \to Y$, $A_0 \to Y$, $L_1 \to A_1 \to Y$, use d-separation to show that conditioning on $L_1$ makes $A_0$ and $U$ dependent.

<details><summary>Show solution</summary>

**Step 1.** $L_1$ is a collider on $A_0 \to L_1 \leftarrow U$.

**Step 2.** Unconditionally the path is blocked ($A_0 \perp U$, since $A_0$ is randomised); conditioning on the collider opens it.

**Step 3.** Since $U \to Y$, the regression coefficient on $A_0$ given $L_1$ picks up a spurious $U$ association. $\blacksquare$

</details>

*Your work:*

### Exercise 9 · Stabilised weights average to one

Show $E[SW] = 1$ with $SW = P(A_1 \mid A_0)/P(A_1 \mid A_0, L_1)$ (time-0 terms cancel under randomisation).

<details><summary>Show solution</summary>

**Step 1.** $E[SW \mid A_0, L_1] = \sum_{a_1}P(a_1 \mid A_0, L_1)\frac{P(a_1 \mid A_0)}{P(a_1 \mid A_0, L_1)} = \sum_{a_1}P(a_1 \mid A_0) = 1$.

**Step 2.** Iterated expectations give $E[SW] = 1$. $\blacksquare$ Mean weights far from 1 signal model misspecification.

</details>

*Your work:*

### Exercise 10 · Counting regimes

With $T$ binary decisions, how many static regimes are there, and why do MSMs impose structure?

<details><summary>Show solution</summary>

**Step 1.** $2^T$ static regimes (1,024 for $T = 10$); dynamic regimes are far more numerous.

**Step 2.** Few patients follow any single sequence, so each regime mean is poorly estimated; an MSM like $\beta_0 + \beta_1\sum_ta_t$ pools information across regimes. $\blacksquare$

</details>

## Part 3 · Applied

*Your work:*

### Exercise 11 · Hypertension treatment over two visits

In a simulation of 20,000 patients, baseline treatment $A_0$ is randomised; doctors intensify treatment $A_1$ more often if blood pressure $L_1$ is still high; unobserved frailty raises both $L_1$ and the cardiovascular risk score $Y$. Each treatment lowers $Y$ by 1. Compare estimators.

<details><summary>Show solution</summary>

**Step 1.** Regression adjusting for $L_1$: effects $-0.64$ ($A_0$) and $-1.02$ ($A_1$): $A_0$ is biased by the collider path.

**Step 2.** Regression without $L_1$: $-0.88$ and $-0.35$: $A_1$ is badly confounded (sicker patients are intensified).

**Step 3.** g-formula: $E[Y(0,0)] = 9.98$, $E[Y(1,1)] = 8.01$ (truth 10 and 8). IPTW MSM: $-0.95$ and $-1.02$.

**Answer:** only the g-methods recover both effects.

</details>

*Your work:*

### Exercise 12 · A dynamic regime

Evaluate "treat at baseline and intensify only if blood pressure is still high" in the same data.

<details><summary>Show solution</summary>

**Step 1.** $P(L_1 = 1 \mid A_0 = 1) = 0.23$.

**Step 2.** The g-formula gives 8.82 (true value 8.78).

**Answer:** close to always-intensify (8.0) only if intensifying helped mainly the high-BP patients; here it helps everyone equally, so always intensifying is better for the risk score, at the cost of intensifying treatment for every patient rather than 23% of them.

</details>

*Your work:*

### Exercise 13 · ART and CD4 counts

In HIV cohorts, low CD4 count prompts antiretroviral therapy and is itself improved by earlier therapy. Why did MSMs change conclusions in these studies?

<details><summary>Show solution</summary>

**Step 1.** Standard regressions either ignored CD4 (confounding by indication: sicker patients treated) or adjusted for it (blocking the benefit operating through CD4).

**Step 2.** MSMs with IPT weights addressed both, revealing a larger protective effect of therapy.

</details>

*Your work:*

### Exercise 14 · Extreme weights

In a cohort study of statin use over 8 years, some cumulative weights exceed 200. What do you do?

<details><summary>Show solution</summary>

**Step 1.** Use stabilised weights and check their mean is near 1.

**Step 2.** Inspect positivity: which covariate histories almost determine treatment?

**Step 3.** Consider truncating at the 1st/99th percentiles (reporting the bias–variance trade-off) or switching to the parametric g-formula.

</details>

*Your work:*

### Exercise 15 · Explaining the problem

Explain to a cardiologist, in two sentences, why adjusting for follow-up blood pressure in a regression gives a misleading treatment effect.

<details><summary>Show solution</summary>

**Step 1.** "Follow-up blood pressure is partly the result of the first treatment, so adjusting for it hides part of that treatment's benefit and compares patients who differ in hidden frailty."

**Step 2.** "But it also drives the second treatment decision, so ignoring it is wrong too; weighting or the g-formula account for it in the right way at each step."

</details>

## Verify in Python

*Your work:*

In [ ]:
import numpy as np

rng = np.random.default_rng(96)
ex = lambda z: 1 / (1 + np.exp(-z))
def sim(n, a0=None, a1=None):
    U = rng.normal(size=n)                                         # unobserved frailty
    A0 = rng.binomial(1, .5, n) if a0 is None else np.full(n, a0)
    L1 = rng.binomial(1, ex(-.5 - A0 + U))                         # high BP at visit 2
    A1 = rng.binomial(1, ex(-1 + 2 * L1)) if a1 is None else np.full(n, a1)
    Y = 10 - A0 - A1 + 2 * U + rng.normal(size=n)                  # cardiovascular risk score
    return A0, L1, A1, Y

truth = {(a, b): round(float(sim(10 ** 6, a, b)[3].mean()), 3) for a in (0, 1) for b in (0, 1)}
print("truth", truth)
n = 20000; A0, L1, A1, Y = sim(n)
ols = lambda Z, y: np.linalg.lstsq(Z, y, rcond=None)[0]
one = np.ones(n)
print("regression with L1   ", ols(np.column_stack([one, A0, A1, L1]), Y)[1:3].round(3))
print("regression without L1", ols(np.column_stack([one, A0, A1]), Y)[1:3].round(3))

m = lambda a, l, b: Y[(A0 == a) & (L1 == l) & (A1 == b)].mean()
pL = {a: L1[A0 == a].mean() for a in (0, 1)}
g = {(a, b): round(float(pL[a] * m(a, 1, b) + (1 - pL[a]) * m(a, 0, b)), 3) for a in (0, 1) for b in (0, 1)}
print("g-formula", g)

pA1 = np.empty(n)
for a in (0, 1):
    for l in (0, 1):
        s = (A0 == a) & (L1 == l); pA1[s] = A1[s].mean()
w = 1 / (.5 * np.where(A1 == 1, pA1, 1 - pA1)); W = np.sqrt(w)
print("IPTW MSM (b0, b_A0, b_A1)", ols(np.column_stack([one, A0, A1]) * W[:, None], Y * W).round(3))

print("dynamic regime (A0=1, A1=L1): g-formula", round(float(pL[1] * m(1, 1, 1) + (1 - pL[1]) * m(1, 0, 0)), 3),
      " P(L1=1|A0=1) =", round(float(pL[1]), 3))
U = rng.normal(size=10 ** 6); L = rng.binomial(1, ex(-1.5 + U))
print("dynamic regime truth", round(float((9 - L + 2 * U).mean()), 3))

## Key takeaways

- With treatment–confounder feedback, regression both with and without the time-varying covariate is biased.
- The g-formula standardises over the covariate's distribution under each regime; IPTW reweights so later treatment is independent of the covariate.
- Sequential exchangeability and positivity are the key assumptions; check weights and consider dynamic regimes for realistic policies.